In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import math
import os
import os.path
import time
import zipfile
import sqlite3

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import plotnine as pn
import scipy
import pycwt

In [3]:
shoreshop_path = os.path.expanduser('~/abmurraylab/shoreshop3')
input_coastsat_path = os.path.join(shoreshop_path, 'inputdata/NC_CoastSat_smoothed.zip')

_dfs = []

with zipfile.ZipFile(input_coastsat_path, 'r') as zipfile:
    for info in zipfile.infolist():
        if not info.filename.startswith('NC_CoastSat_smoothed/usa_NC_'):
            continue
        transect_id = info.filename.removeprefix('NC_CoastSat_smoothed/usa_NC_').removesuffix('.csv')

        with zipfile.open(info.filename) as file:
            _df = pd.read_csv(file, names=('date', 'x'))
            _df['date'] = pd.to_datetime(_df['date'])
            _df['transect'] = transect_id
            _dfs.append(_df)

input_coastsat = pd.concat(_dfs)
input_coastsat

KeyboardInterrupt: 

In [6]:
submissions_path = os.path.join(shoreshop_path, 'submissions/UserSubmissionsCleaned')

nc_cur_paths = {
    'BRIE': os.path.join(submissions_path, 'BRIE/mipNC_1980-2023-BRIE.csv'),
    'CCOST': os.path.join(submissions_path, 'CCOST/mipNC_1980-2023_DESM-reconstruction-CCOST.csv'),
    'CEERD-CHL-Devoe': os.path.join(submissions_path, 'CEERD-CHL-Devoe/mipNC_1980-2023_waves_simplified_ShoreFor_BruunSLR_CUDEMslope_ERDCCHL_SRD.csv'),
    'COCollective': os.path.join(submissions_path, 'COCollective/mipNC_1980-2023_CenturyHindcast-COCollective.csv'),
    'CoSMoS-COAST-conv_IA': os.path.join(submissions_path, 'CoSMoS-COAST_SV/NC/CoSMoS_COAST_conv_model_with_interannual/mipNC_1980-2023_CoSMoS_COAST_conv_model_with_interannual_SV.csv'),
    'CoSMoS-COAST-conv': os.path.join(submissions_path, 'CoSMoS-COAST_SV/NC/CoSMoS_COAST_conv_model_without_interannual/mipNC_1980-2023_CoSMoS_COAST_conv_model_without_interannual_SV.csv'),
    'CoSMoS-COAST-Vitousek_SciReports_2024': os.path.join(submissions_path, 'CoSMoS-COAST_SV/NC/CoSMoS_COAST_Vitousek_SciReports_2024_model/mipNC_1980-2023_CoSMoS_COAST_Vitousek_SciReports_2024_model_SV.csv'),
    'CoSMoS-COAST-CoastSat_trends': os.path.join(submissions_path, 'CoSMoS-COAST_SV/NC/CoastSat_trends/mipNC_1980-2023_CoastSat_trends_SV.csv'),
    'CoSMoS-COAST-CoastSat_trends_bruun_ssp585': os.path.join(submissions_path, 'CoSMoS-COAST_SV/NC/CoastSat_trends_plus_bruun_ssp585/mipNC_1980-2023_CoastSat_trends_plus_bruun_ssp585_SV.csv'),
    'CoSMoS-COAST-DSAS_trends': os.path.join(submissions_path, 'CoSMoS-COAST_SV/NC/DSAS_trends/mipNC_1980-2023_DSAS_SV.csv'),
    'CoSMoS-COAST-persistence': os.path.join(submissions_path, 'CoSMoS-COAST_SV/NC/persistence/mipNC_1980-2023_persistence_SV.csv'),
    'CRCHI': os.path.join(submissions_path, 'CRCHI/mipNC_1980-2023_waves_simplified_smoothed_shorelines-CRCHI.csv'),
    'GEOOCEAN-cross_eps': os.path.join(submissions_path, 'GEOOCEAN/mipNC_1980-2023_GeoOcean_cross_eps_bulk_SLP_Qs_partitions_N.csv'),
    'ICM-TCN': os.path.join(submissions_path, 'ICM-TCN/mipNC_1980-2023_ICM-TCN_eof.csv'),
    'IdSOR': os.path.join(submissions_path, 'IdSOR/mipNC_1980-2023_WaveBulkParameters_IdSOR.csv'),
    'mtraboulsi689': os.path.join(submissions_path, 'mtraboulsi689/mipNC_1980-2023_waves_simplified-MohammadTraboulsi.csv'),
    'UNSW-WRL-Mao': os.path.join(submissions_path, 'UNSW-WRL-Mao/shoreshop3NC/mipNC_1980-2023_GraphShore.csv'),
}

print(len(nc_cur_paths))

start = time.perf_counter_ns()

conn = sqlite3.connect('/scratch/model-data.sqlite')

conn.execute('pragma foreign_keys = on')
conn.execute('pragma cache_size = -4000000')
conn.execute('pragma mmap_size = 4000000000')

conn.executescript('''
create table if not exists transect(
    id integer primary key,
    name text unique not null
);

create table if not exists model(
    id integer primary key,
    name text unique not null
);

create table if not exists data(
    model_id integer references model not null,
    transect_id integer references transect not null,
    date real not null, -- julianday
    x real,
    primary key(model_id, transect_id, date)
) without rowid;
''')

def fix_date(date: str) -> str:
    if '/' not in date:
        return date
    m, d, y = list(map(int, date.split('/')))
    return f'{y:04}-{m:02}-{d:02}'

dfs = []
for model, path in nc_cur_paths.items():
    print(model)
    with open(path) as file:
        header = next(file)
        transects = [col.removeprefix('usa_NC_') for col in header.rstrip().split(',')[1:]]
        transect_ids = []
        with conn:
            row = conn.execute('select id from model where name = ?', (model, )).fetchone()
            if row is None:
                model_id = conn.execute('insert or ignore into model (name) values (?)', (model, )).lastrowid
            else:
                continue
            for transect in transects:
                conn.execute('insert or ignore into transect (name) values (?)', (transect, ))
                id, = conn.execute('select id from transect where name = ?', (transect, )).fetchone()
                transect_ids.append(id)

            for line in file:
                values = line.rstrip().split(',')
                date = fix_date(values[0])
                rows = [
                    (model_id, transect_id, date, None if x == '' else float(x))
                    for transect_id, x in zip(transect_ids, values[1:])
                ]
                conn.executemany('''
                    insert or ignore into data (model_id, transect_id, date, x)
                    values (?, ?, julianday(?), ?)
                ''', rows)

    #df = pd.read_csv(path)
    #columns = {df.columns[0]: 'date'}
    #for column in df.columns.values[1:]:
    #    columns[column] = column.removeprefix('usa_NC_')
    #df = df.rename(columns=columns)
    #df = pd.melt(df, id_vars=['date'], var_name='transect', value_name='x')
    #df['model'] = model
    #dfs.append(df)

#nc_cur = pd.concat(dfs)
#nc_cur['date'] = nc_cur['date'].apply(fix_date)

elapsed = time.perf_counter_ns() - start
print(f'{elapsed/1e6:.3f} ms')

#nc_cur

print(conn.execute('select count(*) from data').fetchone()[0])

# Takes about 10 minutes to generate the database, 10-20 seconds to copy it to/from /scratch, 15 ms to read a model+transect pair

17
BRIE
CCOST
CEERD-CHL-Devoe
COCollective
CoSMoS-COAST-conv_IA
CoSMoS-COAST-conv
CoSMoS-COAST-Vitousek_SciReports_2024
CoSMoS-COAST-CoastSat_trends
CoSMoS-COAST-CoastSat_trends_bruun_ssp585
CoSMoS-COAST-DSAS_trends
CoSMoS-COAST-persistence
CRCHI
GEOOCEAN-cross_eps
ICM-TCN
IdSOR
mtraboulsi689
UNSW-WRL-Mao
589209.848 ms
278799708


In [8]:
def load_model_transect_data(conn: sqlite3.Connection, model: str, transect: str) -> pd.DataFrame:
    return pd.read_sql_query('''
        select model.name as model, transect.name as transect, date(date), x
        from data
            join model on data.model_id = model.id
            join transect on data.transect_id = transect.id
        where model.name = :model and transect.name = :transect
    ''', conn, params={'model': model, 'transect': transect})

In [12]:
start = time.perf_counter_ns()

df = load_model_transect_data(conn, 'IdSOR', '0001_0001')

elapsed = time.perf_counter_ns() - start
print(f'{elapsed/1e6:.3f} ms')

df

14.307 ms


,model,transect,date(date),x
0,IdSOR,0001_0001,1980-01-01,None
1,IdSOR,0001_0001,1980-01-02,None
2,IdSOR,0001_0001,1980-01-03,None
3,IdSOR,0001_0001,1980-01-04,None
4,IdSOR,0001_0001,1980-01-05,None
...,...,...,...,...
16066,IdSOR,0001_0001,2023-12-27,None
16067,IdSOR,0001_0001,2023-12-28,None
16068,IdSOR,0001_0001,2023-12-29,None
16069,IdSOR,0001_0001,2023-12-30,None
